# CKD Data Preprocessing
## StandardScaler + OneHotEncoder

**Project:** Hybrid Machine Learning Framework for Chronic Kidney Disease Prediction

### Methods used
- **Numerical features:** Median Imputation → **StandardScaler**
- **Categorical features:** Most-Frequent Imputation → **OneHotEncoder**
- **Target:** `class` — not scaled as an input feature

> **Data integrity:** The original CSV file is not modified. All transformations are fitted on the training data and applied to separate transformed representations.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Load the dataset
file_path = "../../dataset/processed/chronic_kidney_disease_processed.csv"
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()

print("Dataset shape:", df.shape)
print("Total missing values:", int(df.isna().sum().sum()))
display(df.head())

Dataset shape: (400, 25)
Total missing values: 0


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,class
0,48.0,80.0,1.020,1.0,0.0,normal,normal,notpresent,notpresent,121.0,...,44.0,7800.0,5.2,yes,yes,no,good,no,no,ckd
1,7.0,50.0,1.020,4.0,0.0,normal,normal,notpresent,notpresent,121.0,...,38.0,6000.0,4.8,no,no,no,good,no,no,ckd
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,...,31.0,7500.0,4.8,no,yes,no,poor,no,yes,ckd
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,...,32.0,6700.0,3.9,yes,no,no,poor,yes,yes,ckd
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,...,35.0,7300.0,4.6,no,no,no,good,no,no,ckd


## 1. Identify Numerical and Categorical Features

The `class` attribute is the target. The remaining attributes are candidate predictors.

In [2]:
target = "class"
X = df.drop(columns=[target])
y = df[target].copy()

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical features:")
print(numeric_features)
print("\nCategorical features:")
print(categorical_features)
print("\nNumerical count:", len(numeric_features))
print("Categorical count:", len(categorical_features))

Numerical features:
['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc']

Categorical features:
['rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']

Numerical count: 14
Categorical count: 10


## 2. Stratified Train-Test Split

An 80:20 stratified split is used. The preprocessing parameters are learned from the training partition only to prevent data leakage.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("\nTraining target distribution:")
display(y_train.value_counts())
print("\nTesting target distribution:")
display(y_test.value_counts())

Training shape: (320, 24)
Testing shape: (80, 24)

Training target distribution:


class
ckd       200
notckd    120
Name: count, dtype: int64


Testing target distribution:


class
ckd       50
notckd    30
Name: count, dtype: int64

## 3. Numerical Feature Transformation — StandardScaler

For numerical attributes:

**Median Imputation → StandardScaler**

StandardScaler standardizes each numerical feature using:

\[
z = \frac{x-\mu}{\sigma}
\]

where the mean and standard deviation are learned from the training data.

In [4]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

print("Numerical pipeline:")
print("Median Imputation → StandardScaler")

Numerical pipeline:
Median Imputation → StandardScaler


## 4. Categorical Feature Transformation — OneHotEncoder

For categorical attributes:

**Most-Frequent Imputation → OneHotEncoder**

OneHotEncoder converts categorical levels into binary indicator variables without imposing an artificial numerical order.

`handle_unknown="ignore"` allows the pipeline to process a previously unseen category safely.

In [5]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

print("Categorical pipeline:")
print("Most-Frequent Imputation → OneHotEncoder")

Categorical pipeline:
Most-Frequent Imputation → OneHotEncoder


## 5. Combined Preprocessing Pipeline

`ColumnTransformer` applies the appropriate transformation according to feature type.

In [6]:
preprocessor = ColumnTransformer([
    ("numerical", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Fit only on training data
X_train_transformed = preprocessor.fit_transform(X_train)

# Apply the fitted transformations to the test data
X_test_transformed = preprocessor.transform(X_test)

print("Original predictor count:", X_train.shape[1])
print("Transformed training columns:", X_train_transformed.shape[1])
print("Transformed testing columns:", X_test_transformed.shape[1])

Original predictor count: 24
Transformed training columns: 34
Transformed testing columns: 34


## 6. Transformed Feature Names

One-hot encoding may increase the number of columns because a single categorical attribute can produce multiple indicator variables.

In [7]:
transformed_feature_names = preprocessor.get_feature_names_out()

print("Number of transformed features:", len(transformed_feature_names))
print("\nTransformed feature names:")
print(transformed_feature_names)

transformed_train_df = pd.DataFrame(
    X_train_transformed,
    columns=transformed_feature_names,
    index=X_train.index
)

display(transformed_train_df.head())

Number of transformed features: 34

Transformed feature names:
['numerical__age' 'numerical__bp' 'numerical__sg' 'numerical__al'
 'numerical__su' 'numerical__bgr' 'numerical__bu' 'numerical__sc'
 'numerical__sod' 'numerical__pot' 'numerical__hemo' 'numerical__pcv'
 'numerical__wbcc' 'numerical__rbcc' 'categorical__rbc_abnormal'
 'categorical__rbc_normal' 'categorical__pc_abnormal'
 'categorical__pc_normal' 'categorical__pcc_notpresent'
 'categorical__pcc_present' 'categorical__ba_notpresent'
 'categorical__ba_present' 'categorical__htn_no' 'categorical__htn_yes'
 'categorical__dm_no' 'categorical__dm_yes' 'categorical__cad_no'
 'categorical__cad_yes' 'categorical__appet_good'
 'categorical__appet_poor' 'categorical__pe_no' 'categorical__pe_yes'
 'categorical__ane_no' 'categorical__ane_yes']


,numerical__age,numerical__bp,numerical__sg,numerical__al,numerical__su,numerical__bgr,numerical__bu,numerical__sc,numerical__sod,numerical__pot,...,categorical__dm_no,categorical__dm_yes,categorical__cad_no,categorical__cad_yes,categorical__appet_good,categorical__appet_poor,categorical__pe_no,categorical__pe_yes,categorical__ane_no,categorical__ane_yes
380,0.468853,-1.313179,0.412298,-0.684861,-0.394777,-0.418417,-0.674210,-0.379844,0.185176,-0.350611,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
56,1.485011,-0.514287,-0.503920,1.622039,3.293647,-0.316916,2.297339,1.520411,-1.075595,-0.063176,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0
126,1.126367,1.083498,-0.503920,-0.684861,-0.394777,-0.025098,1.475421,0.260940,-0.287613,0.000699,...,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
371,-1.384143,-1.313179,1.328515,-0.684861,-0.394777,-0.849800,-0.105190,-0.512420,1.130754,0.128448,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
333,-1.683013,0.284605,0.412298,-0.684861,-0.394777,-0.596046,-0.189489,-0.357748,0.657965,-0.190925,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0


## 7. Verify StandardScaler

The numerical training features should have approximately zero mean and unit standard deviation after scaling.

In [8]:
# Numerical transformer after fitting
fitted_numeric_pipeline = preprocessor.named_transformers_["numerical"]

scaled_numeric = fitted_numeric_pipeline.transform(X_train[numeric_features])
scaled_numeric_df = pd.DataFrame(scaled_numeric, columns=numeric_features)

verification = pd.DataFrame({
    "Mean": scaled_numeric_df.mean(),
    "Std": scaled_numeric_df.std(ddof=0)
})

display(verification.round(4))

,Mean,Std
age,0.0,1.0
bp,0.0,1.0
sg,0.0,1.0
al,-0.0,1.0
su,-0.0,1.0
bgr,-0.0,1.0
bu,-0.0,1.0
sc,-0.0,1.0
sod,0.0,1.0
pot,0.0,1.0


## 8. Verify One-Hot Encoding

In [9]:
fitted_categorical_pipeline = preprocessor.named_transformers_["categorical"]
onehot = fitted_categorical_pipeline.named_steps["onehot"]

encoded_names = onehot.get_feature_names_out(categorical_features)

print("Number of one-hot encoded categorical columns:", len(encoded_names))
print("\nEncoded categorical columns:")
print(encoded_names)

Number of one-hot encoded categorical columns: 20

Encoded categorical columns:
['rbc_abnormal' 'rbc_normal' 'pc_abnormal' 'pc_normal' 'pcc_notpresent'
 'pcc_present' 'ba_notpresent' 'ba_present' 'htn_no' 'htn_yes' 'dm_no'
 'dm_yes' 'cad_no' 'cad_yes' 'appet_good' 'appet_poor' 'pe_no' 'pe_yes'
 'ane_no' 'ane_yes']


## 9. Original Dataset Integrity Check

The source dataframe is retained separately from the transformed modelling matrix.

**The original dataset is not overwritten.**

- Original rows: 400
- Original columns: 25
- Original predictor variables: 24
- Target: `class`

The transformed matrix is only a computational representation used by downstream machine-learning models.

In [10]:
print("Original dataset shape:", df.shape)
print("Original column count:", len(df.columns))
print("Original missing-cell count:", int(df.isna().sum().sum()))
print("\nOriginal columns:")
print(df.columns.tolist())

# The transformed matrices are separate objects
print("\nTransformed training matrix shape:", X_train_transformed.shape)
print("Transformed testing matrix shape:", X_test_transformed.shape)

Original dataset shape: (400, 25)
Original column count: 25
Original missing-cell count: 0

Original columns:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'class']

Transformed training matrix shape: (320, 34)
Transformed testing matrix shape: (80, 34)


## 10. Final Preprocessing Specification

| Feature type | Missing-value treatment | Transformation |
|---|---|---|
| Numerical | Median imputation | **StandardScaler** |
| Categorical | Most-frequent imputation | **OneHotEncoder** |
| Target `class` | Not used as an input feature | **No scaling** |

### Final workflow

**Original CKD Dataset → Stratified Train/Test Split → Numerical Imputation + StandardScaler / Categorical Imputation + OneHotEncoder → ML Models**

This preprocessing is performed without changing the original source dataset.

## Conclusion

The CKD dataset is transformed using a type-specific preprocessing strategy:

- **StandardScaler** for numerical predictors
- **OneHotEncoder** for categorical predictors
- **Median imputation** for missing numerical values
- **Most-frequent imputation** for missing categorical values

The preprocessing is fitted only on the training partition, helping prevent data leakage. The original CSV remains unchanged.